## Evaluación Parcial N°2

Asignatura: Preprocesamiento de Datos

Integrantes: 
* Alejandra González
* Constanza González
* Diego Villar

Sección: 800D

In [2]:
import sys
!{sys.executable} -m pip install matplotlib seaborn

## [Celda 1] Carga de librerías y lectura del dataset

In [4]:
# Importación de librerías base para la manipulación de datos
import pandas as pd
import numpy as np

# Carga del dataset original
# Se especifica el separador ';' ya que el archivo no usa comas estándar
df = pd.read_csv('bank-additional-full.csv', sep=';')

# Verificamos la forma inicial del dataset y las primeras filas
print(f"Dimensiones iniciales: {df.shape}")
df.head()

Dimensiones iniciales: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## [Celda 2] Estandarización de formatos y traducción

In [5]:
# Limpieza de strings: quitamos el punto en la categoría 'admin.' para estandarizar
df['job'] = df['job'].str.replace('admin.', 'admin', regex=False)

# Diccionarios de traducción para mejorar la presentación al cliente en los gráficos
meses_es = {
    'may': 'mayo', 'jun': 'junio', 'jul': 'julio', 'aug': 'agosto', 
    'oct': 'octubre', 'nov': 'noviembre', 'dec': 'diciembre', 
    'mar': 'marzo', 'apr': 'abril', 'sep': 'septiembre'
}
dias_es = {
    'mon': 'lunes', 'tue': 'martes', 'wed': 'miércoles', 
    'thu': 'jueves', 'fri': 'viernes'
}

# Aplicamos la traducción a las columnas correspondientes
df['month'] = df['month'].map(meses_es)
df['day_of_week'] = df['day_of_week'].map(dias_es)

print("Traducción y estandarización completada.")

Traducción y estandarización completada.


## [Celda 3] Tratamiento de valores "unknown" (Datos Faltantes)

In [6]:
# Variables con valores 'unknown' detectadas: job, marital, education, default, housing, loan

# Estrategia: Imputación por la moda (el valor más frecuente)
# Para no perder filas valiosas, reemplazamos 'unknown' por el valor más común de cada columna.
columnas_con_unknown = ['job', 'marital', 'education', 'default', 'housing', 'loan']

for col in columnas_con_unknown:
    # Encontramos la moda de la columna (excluyendo 'unknown')
    moda_columna = df[df[col] != 'unknown'][col].mode()[0]
    
    # Reemplazamos 'unknown' con la moda calculada
    df[col] = df[col].replace('unknown', moda_columna)

print("Valores 'unknown' imputados correctamente mediante la moda estadística.")

Valores 'unknown' imputados correctamente mediante la moda estadística.


## [Celda 4] Reducción de cardinalidad

In [7]:
# La variable education tiene mucha fragmentación en la educación básica
# Agrupamos 'basic.4y', 'basic.6y' y 'basic.9y' en una sola categoría: 'basic'
categorias_basicas = ['basic.4y', 'basic.6y', 'basic.9y']
df['education'] = df['education'].replace(categorias_basicas, 'basic')

# Verificamos el cambio
print("Nuevas categorías en educación:")
print(df['education'].unique())

Nuevas categorías en educación:
<StringArray>
[              'basic',         'high.school', 'professional.course',
   'university.degree',          'illiterate']
Length: 5, dtype: str


## [Celda 5] Transformación de variables problemáticas y Ética de datos

In [8]:
# 1. Tratamiento del outlier estructural en pdays (999 significa no contactado previamente)
# Creamos una variable binaria que aporte más valor predictivo/analítico
df['contactado_previamente'] = np.where(df['pdays'] == 999, 0, 1)

# Reemplazamos los 999 por NaN en pdays para que no arruinen promedios si se decide usar la variable numérica
df['pdays'] = df['pdays'].replace(999, np.nan)

# 2. Eliminación de la variable 'duration' (Decisión de negocio y ética)
# Como se indica en el caso, afecta la variable objetivo pero no se conoce antes de la llamada, 
# creando un modelo irrealista. La eliminamos del dataset que se usará para modelar.
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print("Variable 'duration' eliminada por ética y viabilidad del modelo predictivo.")

Variable 'duration' eliminada por ética y viabilidad del modelo predictivo.


## [Celda 6] Tratamiento de Outliers (Límite en llamadas por campaña)

In [9]:
# La variable 'campaign' tiene clientes con un número extremo de contactos.
# Usaremos el Rango Intercuartílico (IQR) para limitar (capping) estos valores extremos 
# y evitar sesgos sin eliminar filas.

Q1 = df['campaign'].quantile(0.25)
Q3 = df['campaign'].quantile(0.75)
IQR = Q3 - Q1

# Definimos el límite superior
limite_superior = Q3 + 1.5 * IQR

# Aplicamos capping: todo valor mayor al límite superior se iguala al límite superior
df['campaign'] = np.where(df['campaign'] > limite_superior, limite_superior, df['campaign'])

print(f"Outliers en 'campaign' limitados a un máximo de {limite_superior} contactos.")

Outliers en 'campaign' limitados a un máximo de 6.0 contactos.


## [Celda 7] Codificación numérica y Exportación

In [10]:
# Transformamos la variable objetivo 'y' ('yes'/'no') a formato binario numérico (1/0)
# Esto es vital para calcular tasas de conversión promedios y matrices de correlación
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Guardamos el dataset limpio en un nuevo archivo CSV
# Este archivo será el input para tu segundo Notebook (el de los gráficos)
df.to_csv('bank_limpio_para_graficos.csv', index=False)

print("Limpieza finalizada. Archivo 'bank_limpio_para_graficos.csv' exportado con éxito.")

Limpieza finalizada. Archivo 'bank_limpio_para_graficos.csv' exportado con éxito.
